# KG Link-Prediction Evaluation

Evaluate the RotatE embeddings produced by `dataprocess.ipynb` on link prediction.

Uses PyKEEN's filtered rank-based evaluator on the held-out test split. Reported metrics:

- **GMR** (geometric mean rank) ↓ lower is better
- **HMR** (harmonic mean rank) ↓ lower is better
- **AMR** (arithmetic mean rank) ↓ lower is better
- **Hits@10** ↑ higher is better

Requires `model.pt`, `train_triples.pt`, and `test_triples.pt` saved by the updated
`dataprocess.ipynb` for each dataset under `../datasets/<dataset>/`.

In [ ]:
import torch

import pandas as pd
from pykeen.evaluation import RankBasedEvaluator

DATASETS = ['ali', 'lastfm', 'mind']

# Metric keys to report (side='both', rank_type='realistic').
METRICS = {
    'GMR': 'geometric_mean_rank',   # lower is better
    'HMR': 'harmonic_mean_rank',    # lower is better
    'AMR': 'arithmetic_mean_rank',  # lower is better
    'Hits@10': 'hits_at_10',        # higher is better
}

In [ ]:
def evaluate_dataset(dataset: str) -> dict[str, float]:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = torch.load(f'../datasets/{dataset}/model.pt', map_location=device, weights_only=False)
    train_triples = torch.load(f'../datasets/{dataset}/train_triples.pt')
    test_triples = torch.load(f'../datasets/{dataset}/test_triples.pt')

    evaluator = RankBasedEvaluator(filtered=True)
    results = evaluator.evaluate(
        model=model,
        mapped_triples=test_triples,
        # filtered setting: a true triple ranked above the target is not penalized
        additional_filter_triples=[train_triples, test_triples],
    )
    return {name: results.get_metric(key) for name, key in METRICS.items()}

## Run evaluation

In [ ]:
results_all = {dataset: evaluate_dataset(dataset) for dataset in DATASETS}

for dataset, metrics in results_all.items():
    print(f'{dataset}:')
    for name, value in metrics.items():
        print(f'  {name:8s} {value:.4f}')

In [ ]:
summary = pd.DataFrame(results_all).T
summary.index.name = 'dataset'
summary